In [ ]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [ ]:
from langchain_core.documents import Document

In [ ]:
# This is just for the understanding doc contains 2 components ie page_content and metadata
sample_doc = Document(
    page_content="Hello World!",
    metadata={"source": "https://www.google.com"}
)
sample_doc

In [ ]:
type(sample_doc)

In [ ]:
# Text data loading
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("data/python.txt", encoding="utf-8")

In [ ]:
document = loader.load()
document

In [ ]:
# PDF data loading 
from langchain_community.document_loaders.pdf import PyMuPDFLoader

pdf_loader = PyMuPDFLoader("data/research.pdf")

In [ ]:
document = pdf_loader.load()
document

# Data ingestion pipeline

In [ ]:
# Remember for this file structure pdfs(research1 and research2) are the vector db we will create 

In [ ]:
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

#### Data converting into documents

In [ ]:
def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []

    for file in os.listdir(folder_path):
        if file.lower().endswith(".pdf"):
            # complete filepath 
            pdf_path = os.path.join(folder_path, file)

            loader = PyPDFLoader(pdf_path)
            document = loader.load()

            all_docs.extend(document)
            num_docs += 1

    print("Total pdfs: ", num_docs)
    print("Total pages: ", len(all_docs))
    return all_docs

In [ ]:
all_pdf_documents = load_all_pdfs()

#### Chunks

In [ ]:
# !pip install langchain_text_splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [ ]:
def split_docs(documents, chunk_size=500, chunk_overlap=50):
    test_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )
    chunked_docs = test_splitter.split_documents(documents)
    return chunked_docs

In [ ]:
chunks = split_docs(all_pdf_documents)
chunks

In [ ]:
len(chunks)

#### Embedding

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
# Creating a class do that we can make objects of it later
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):

        self.model_name = model_name
        print("Loading Model: ", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("Embedding Dimensions: ", self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("Embeddings shape: ", embeddings.shape)
        return embeddings

In [ ]:
embedding_manager = EmbeddingManager()

#### Vector DB

In [ ]:
# imp we will create vector store ie nothing but a interface used to interact with vector db
import chromadb
import uuid

In [ ]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.persist_directory = persist_directory
        self.collection_name = collection_name
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)

        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)
        
        # create a collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"Description": "Vector store collection for pdf embeddings in RAG"}
        )
        print("Initialized the vector store with collection: ", self.collection_name)
        print("Docs in collection: ", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents does not match number of embeddings")

        # store => ids, embeddings, documents, metadata
        ids=[]
        all_metadata=[]
        documents_content=[]
        embeddings_list=[]

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            # Add all in the collection
        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embeddings_list
        )
        print("Total documents added in vector store: ", len(documents_content))
        print("Docs in collection: ", self.collection.count())

In [ ]:
vector_store = VectorStoreManager()

In [ ]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embedding)

# Retrieval pipeline

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]
        
        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs = []

        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance # this is formula

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "metadata": metadata,
                        "document": document,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": i + 1
                    })
            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("No documents found")

        return retrieved_docs

In [ ]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [ ]:
rag_retriever.retrieve("What is RAG")

# RAG with GEMINI

In [ ]:
# !pip install langchain langchain-google-genai google-generativeai python-dotenv

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

google_api_key = os.getenv("GOOGLE_API_KEY")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",  
    google_api_key=google_api_key,
    temperature=0.1,
    max_output_tokens=1024
)

In [ ]:
# generate our retrieval-augmented o/p
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("We found no relevant context for the given query")

    # Context + query = prompt
    prompt = f""" use given context to generate the answer for the query
                  Context: {context}
                  Query:{query}"""

    response = llm.invoke([prompt.format(context=context, query=query)]) #expected a list as prompt
    return response.content

In [ ]:
answer = generate_output("What is RAG??", rag_retriever, llm)

In [ ]:
print(answer)

In [ ]:
# !pip install google-generativeai

In [ ]:
import os
import google.generativeai as genai

# Configure Gemini
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

# Load model
model = genai.GenerativeModel("gemini-2.5-flash-lite")

# Generate response
response = model.generate_content("What is a+2=8?")

print(response.text)

In [ ]:
response

In [ ]:
# Image as i/p

import requests
import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

model = genai.GenerativeModel("gemini-2.5-flash-lite")

# Image URL
url = "https://t3.ftcdn.net/jpg/18/74/38/36/240_F_1874383619_ZyEbxWYcKb6Dvonnh71mVd4spuuOHMNh.jpg"

# Download image
img_data = requests.get(url).content

# Gemini input (text + image together)
response = model.generate_content([
    "Tell me the capital of this flag's country?",
    {
        "mime_type": "image/jpeg",
        "data": img_data
    }
])

print(response.text)

# AI personal assistant

In [ ]:
class PersonalAssistant:
    def __init__(self):
        self.model = genai.GenerativeModel(
            model_name="gemini-2.5-flash-lite",
            system_instruction="Act like a helpful personal assistant"
        )
        self.chat = self.model.start_chat(history=[])
        print("Hi, I am your AI assistant. How can I help?")

    def ans_query(self):
        question = input("Ask me anything: ")
        response = self.chat.send_message(question)
        print(response.text.strip())

In [ ]:
assistant = PersonalAssistant()

In [ ]:
assistant.ans_query()

In [ ]:
# !pip install requests pillow python-dotenv

In [ ]:
# Image as o/p using hugging face
import requests
import os
from PIL import Image
from io import BytesIO
from IPython.display import display
from dotenv import load_dotenv

load_dotenv()
huggingface_api_key = os.getenv("HUGGINGFACE_API_KEY")

API_URL = "https://router.huggingface.co/hf-inference/models/black-forest-labs/FLUX.1-schnell"

headers = {"Authorization": f"Bearer {huggingface_api_key}"}

prompt = "A highly detailed futuristic spaceship flying near a nebula, hyper-realistic, digital art, 4k"

print("Generating image...")
response = requests.post(API_URL, headers=headers, json={"inputs": prompt})

if response.status_code == 200:
    img = Image.open(BytesIO(response.content))
    img.save("spaceship.png")
    display(img)
    print("Image saved as spaceship.png!")
else:
    print(f"Error {response.status_code}: {response.text}")